In [15]:
#!/usr/bin/env python3
"""
LAB CLASE S5 — MediSalud S.A. — Auditoría de Código
Nombre del estudiante:DALIA NANCY LEON AGUILAR
Fecha:18 JULIO 2026
"""

# ═══════════════════════════════════════════════════
# FRAGMENTO 1 — Sistema de búsqueda de pacientes
# ═══════════════════════════════════════════════════

def buscar_paciente_INSEGURO(nombre_paciente: str) -> str:
    """Versión INSEGURA — para análisis."""
    # Simula la consulta SQL que se ejecutaría en BD real
    query = f"SELECT * FROM pacientes WHERE nombre LIKE '%{nombre_paciente}%'"
    return f"[Ejecutando]: {query}"

def buscar_paciente_SEGURO(nombre_paciente: str) -> str:
    """
    ▶ TODO 1: Implementar versión segura.
    Usa consulta parametrizada y valida el input.
    La consulta parametrizada se vería así:
    query = "SELECT * FROM pacientes WHERE nombre LIKE ?"
    valor = (f"%{nombre_paciente}%",)
    """
    # Validación del input
    if not nombre_paciente or len(nombre_paciente) > 100:
        return "Error: nombre inválido"

    # ▶ COMPLETA AQUÍ: escribe la consulta parametrizada
    query_segura = "SELECT * FROM pacientes WHERE nombre LIKE ?"  # Reemplazar con la consulta correcta
    valor = (f"%{nombre_paciente}%",)       # Reemplazar con el valor parametrizado

    # Simula la ejecución (en lab real usaríamos cursor.execute(query_segura, valor))
    return f"[Ejecutando de forma segura]: {query_segura} con valor={valor}"


# ═══════════════════════════════════════════════════
# FRAGMENTO 2 — Login del personal médico
# ═══════════════════════════════════════════════════

def login_personal_INSEGURO(usuario: str, password: str) -> dict:
    """Versión INSEGURA — para análisis."""
    query = "SELECT id, nombre, especialidad FROM medicos WHERE usuario='" + usuario + "' AND password='" + password + "'"

    # Simula la ejecución (en lab real: cursor.execute(query))
    print(f"  SQL generado: {query}")

    # Simula resultado (asume que el payload admin'-- devuelve resultado)
    if "OR '1'='1" in query or "--" in query:
        return {"acceso": True, "usuario": "BYPASS_EXITOSO", "via": "SQL_INJECTION"}
    return {"acceso": False}

def login_personal_SEGURO(usuario: str, password: str) -> dict:
    """
    ▶ TODO 2: Implementar versión segura.
    1. Validar que usuario solo tenga caracteres alfanuméricos y .
    2. Usar prepared statement
    3. Devolver mensaje genérico si falla (no revelar cuál campo falló)
    """
    import re

    # ▶ COMPLETA: validación del usuario
    if not re.match(r'^[a-zA-Z0-9.]{3,50}$', usuario):  # Regex que permita solo letras, números y punto
        return {"acceso": False, "mensaje": "Datos inválidos"}

    # Simula la consulta parametrizada
    query_segura = "SELECT id, nombre, especialidad FROM medicos WHERE usuario=? AND password=?"
    print(f"  SQL parametrizado: {query_segura}")
    print(f"  Parámetros: ({repr(usuario)}, [PROTEGIDO])")

    # Simula verificación (siempre falla en el lab porque no tenemos BD real)
    return {"acceso": False, "mensaje": "Credenciales inválidas"}  # Mensaje genérico


# ═══════════════════════════════════════════════════
# FRAGMENTO 3 — Endpoint de diagnóstico del servidor
# ═══════════════════════════════════════════════════

def diagnostico_servidor_INSEGURO(ip_host: str) -> str:
    """Versión INSEGURA — para análisis."""
    import subprocess
    # VULNERABILIDAD: shell=True + concatenación = Command Injection
    cmd = f"ping -c 2 {ip_host}"
    resultado = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return resultado.stdout

def diagnostico_servidor_SEGURO(ip_host: str) -> str:
    """
    ▶ TODO 3: Implementar versión segura.
    1. Validar que ip_host sea una IP válida con ipaddress.ip_address()
    2. Usar subprocess con LISTA de argumentos (sin shell=True)
    3. Rechazar IPs privadas
    """
    import subprocess
    import ipaddress

    try:
        # ▶ COMPLETA: validar la IP
        ip = ipaddress.ip_address(ip_host)  # Pasar ip_host aquí

        # ▶ COMPLETA: rechazar IPs privadas/loopback
        if ip.is_private:  # Añadir también is_loopback
            return "Error: IP no permitida"

        # ▶ COMPLETA: subprocess con lista (SIN shell=True)
        resultado = subprocess.run(
            ["ping", "-c", "2", str(ip)],  # Pasar str(ip) aquí
            capture_output=True, text=True, timeout=5
        )
        return resultado.stdout

    except ValueError:
        return "Error: IP inválida"


# ═══════════════════════════════════════════════════
# PRUEBAS — Ejecuta esto para verificar tu trabajo
# ═══════════════════════════════════════════════════

if __name__ == "__main__":
    print("=" * 60)
    print("AUDITORÍA DE SEGURIDAD — MediSalud S.A.")
    print("=" * 60)

    # ─── FRAGMENTO 1: Búsqueda de pacientes ───
    print("\n▌ FRAGMENTO 1 — Búsqueda de pacientes")
    print("\n  [INSEGURO] Búsqueda normal:")
    print(" ", buscar_paciente_INSEGURO("Garcia"))

    print("\n  [INSEGURO] Con SQL Injection:")
    payload_sqli = "' UNION SELECT usuario, password, NULL FROM medicos--"
    print(" ", buscar_paciente_INSEGURO(payload_sqli))

    print("\n  [SEGURO] Misma búsqueda normal:")
    print(" ", buscar_paciente_SEGURO("Garcia"))

    print("\n  [SEGURO] Intento de SQL Injection (debe ser bloqueado):")
    print(" ", buscar_paciente_SEGURO(payload_sqli))

    # ─── FRAGMENTO 2: Login del personal ───
    print("\n" + "=" * 60)
    print("▌ FRAGMENTO 2 — Login del personal médico")
    print("\n  [INSEGURO] Login normal:")
    print(" ", login_personal_INSEGURO("dr.garcia", "clave123"))

    print("\n  [INSEGURO] Con SQL Injection:")
    print(" ", login_personal_INSEGURO("admin'--", "cualquiercosa"))

    print("\n  [SEGURO] Intento de bypass (debe ser rechazado):")
    print(" ", login_personal_SEGURO("admin'--", "cualquiercosa"))

    # ─── FRAGMENTO 3: Diagnóstico ───
    print("\n" + "=" * 60)
    print("▌ FRAGMENTO 3 — Diagnóstico del servidor")
    print("\n  [INSEGURO - SOLO MOSTRAR, NO EJECUTAR EN CLASE]")
    print("  Payload: '127.0.0.1; ls -la' ejecutaría comandos del SO")

    print("\n  [SEGURO] IP válida pública:")
    try:
      print(" ", diagnostico_servidor_SEGURO("8.8.8.8"))
    except Exception as e:
      print("  (no se pudo ejecutar ping en este entorno, pero la IP pasó la validación correctamente)")

    print("\n  [SEGURO] IP privada (debe rechazarse):")
    print(" ", diagnostico_servidor_SEGURO("192.168.1.1"))

    print("\n  [SEGURO] Command injection (debe rechazarse):")
    print(" ", diagnostico_servidor_SEGURO("8.8.8.8; ls -la"))

    # ─── PREGUNTA FINAL ───
    print("\n" + "=" * 60)
    print("▌ PREGUNTA DE ANÁLISIS FINAL")
    print("""
    Responde directamente en este archivo (agrega un comentario al final):

    1. ¿Cuál de los 3 fragmentos tiene la vulnerabilidad más peligrosa? ¿Por qué?

    2. ¿Qué tienen en común las 3 vulnerabilidades a nivel de causa raíz?

    3. ¿Cómo verificarías que la versión SEGURA realmente funciona
       (qué pruebas harías)?
    """)

# ═══════════════════════════════════════════════════
# TUS RESPUESTAS (agrega aquí):
# ═══════════════════════════════════════════════════
"""
1. El fragmento más peligroso es: el Fragmento 3 (Command Injection)
   Porque: el SQL Injection permite leer o alterar datos dentro de la base de datos,
   pero el Command Injection le da al atacante ejecución de comandos directamente sobre
   el sistema operativo del servidor. Con eso puede leer
   cualquier archivo, robar la base de datos completa, instalar backdoors o
   moverse hacia otros sistemas de la red — el impacto no queda limitado a la
   BD, compromete todo el servidor.


2. Lo que tienen en común es: el desarrollador tomó un dato que viene directamente
   del usuario (sin validar ni separar del código) y lo concatenó dentro de un
   comando que un intérprete (SQL o el shell del sistema operativo) ejecuta literalmente.
   La causa raíz es la misma:
   mezclar datos no confiables con la sintaxis del comando/consulta en vez de
   tratarlos siempre como datos puros (parámetros).


3. Las pruebas que haría son:
   - Reenviar los mismos payloads maliciosos usados en las versiones
     INSEGURAS y confirmar que la versión SEGURA los rechaza o los trata como
     texto literal (no como código SQL/shell).
   - Probar con datos límite (nombre de 101 caracteres, usuario con símbolos
     no permitidos, IP mal formada) para confirmar que las validaciones
     rechazan correctamente.
   - Revisar que el mensaje de error nunca exponga la consulta SQL generada
     ni detalles internos del sistema.
   - Confirmar, leyendo el código, que en ningún punto se use f-string o
     concatenación directa dentro de la consulta o del comando ejecutado.
"""



AUDITORÍA DE SEGURIDAD — MediSalud S.A.

▌ FRAGMENTO 1 — Búsqueda de pacientes

  [INSEGURO] Búsqueda normal:
  [Ejecutando]: SELECT * FROM pacientes WHERE nombre LIKE '%Garcia%'

  [INSEGURO] Con SQL Injection:
  [Ejecutando]: SELECT * FROM pacientes WHERE nombre LIKE '%' UNION SELECT usuario, password, NULL FROM medicos--%'

  [SEGURO] Misma búsqueda normal:
  [Ejecutando de forma segura]: SELECT * FROM pacientes WHERE nombre LIKE ? con valor=('%Garcia%',)

  [SEGURO] Intento de SQL Injection (debe ser bloqueado):
  [Ejecutando de forma segura]: SELECT * FROM pacientes WHERE nombre LIKE ? con valor=("%' UNION SELECT usuario, password, NULL FROM medicos--%",)

▌ FRAGMENTO 2 — Login del personal médico

  [INSEGURO] Login normal:
  SQL generado: SELECT id, nombre, especialidad FROM medicos WHERE usuario='dr.garcia' AND password='clave123'
  {'acceso': False}

  [INSEGURO] Con SQL Injection:
  SQL generado: SELECT id, nombre, especialidad FROM medicos WHERE usuario='admin'--' AND passwo

'\n1. El fragmento más peligroso es: el Fragmento 3 (Command Injection)\n   Porque: el SQL Injection permite leer o alterar datos dentro de la base de datos, \n   pero el Command Injection le da al atacante ejecución de comandos directamente sobre \n   el sistema operativo del servidor. Con eso puede leer\n   cualquier archivo, robar la base de datos completa, instalar backdoors o\n   moverse hacia otros sistemas de la red — el impacto no queda limitado a la\n   BD, compromete todo el servidor.\n\n\n2. Lo que tienen en común es: el desarrollador tomó un dato que viene directamente \n   del usuario (sin validar ni separar del código) y lo concatenó dentro de un \n   comando que un intérprete (SQL o el shell del sistema operativo) ejecuta literalmente.\n   La causa raíz es la misma:\n   mezclar datos no confiables con la sintaxis del comando/consulta en vez de\n   tratarlos siempre como datos puros (parámetros).\n\n\n3. Las pruebas que haría son: \n   - Reenviar los mismos payloads malic

In [10]:
!apt-get install -y iputils-ping -q

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  iputils-ping
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 43.0 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 iputils-ping amd64 3:20211215-1ubuntu0.1 [43.0 kB]
Fetched 43.0 kB in 0s (119 kB/s)
Selecting previously unselected package iputils-ping.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../iputils-ping_3%3a20211215-1ubuntu0.1_amd64.deb ...
Unpacking iputils-ping (3:20211215-1ubuntu0.1) ...
Setting up iputils-ping (3:20211215-1ubuntu0.1) ...
Processing triggers for man-db (2.10.2-1) ...


In [16]:
import sqlite3

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.execute("CREATE TABLE pacientes (id INTEGER PRIMARY KEY, nombre TEXT)")
cursor.execute("CREATE TABLE medicos (usuario TEXT, password TEXT)")
cursor.execute("INSERT INTO pacientes (nombre) VALUES (?)", ("Garcia",))
cursor.execute("INSERT INTO pacientes (nombre) VALUES (?)", ("Perez",))
cursor.execute("INSERT INTO medicos VALUES (?, ?)", ("admin", "adminpass"))

# Inseguro
def buscar_pacientes_INSEGURO(nombre):
    sql = f"SELECT * FROM pacientes WHERE nombre LIKE '%{nombre}%'"
    print("[INSEGURO] SQL generado:", sql)
    return cursor.execute(sql).fetchall()

# Seguro
def buscar_pacientes_SEGURO(nombre):
    sql = "SELECT * FROM pacientes WHERE nombre LIKE ?"
    valor = f"%{nombre}%"
    print("[SEGURO] SQL generado:", sql, "con valor=", (valor,))
    return cursor.execute(sql, (valor,)).fetchall()

print(buscar_pacientes_INSEGURO("Garcia"))
print(buscar_pacientes_INSEGURO("' UNION SELECT usuario, password FROM medicos--"))
print(buscar_pacientes_SEGURO("Garcia"))
print(buscar_pacientes_SEGURO("' UNION SELECT usuario, password FROM medicos--"))



[INSEGURO] SQL generado: SELECT * FROM pacientes WHERE nombre LIKE '%Garcia%'
[(1, 'Garcia')]
[INSEGURO] SQL generado: SELECT * FROM pacientes WHERE nombre LIKE '%' UNION SELECT usuario, password FROM medicos--%'
[(1, 'Garcia'), (2, 'Perez'), ('admin', 'adminpass')]
[SEGURO] SQL generado: SELECT * FROM pacientes WHERE nombre LIKE ? con valor= ('%Garcia%',)
[(1, 'Garcia')]
[SEGURO] SQL generado: SELECT * FROM pacientes WHERE nombre LIKE ? con valor= ("%' UNION SELECT usuario, password FROM medicos--%",)
[]


In [17]:
!pip install bcrypt
import bcrypt
import re
import logging



logging.basicConfig(level=logging.ERROR)

cursor.execute("CREATE TABLE IF NOT EXISTS medicos (usuario TEXT, password TEXT)")
hashed = bcrypt.hashpw("clave123".encode(), bcrypt.gensalt())
cursor.execute("INSERT INTO medicos VALUES (?, ?)", ("dr.garcia", hashed.decode()))

def login_personal_SEGURO(usuario, clave):
    try:
        # Validación simple de caracteres permitidos
        if not re.match(r"^[a-zA-Z0-9_.-]+$", usuario):
            return {"acceso": False, "error": "Usuario inválido"}

        cursor.execute("SELECT usuario, password FROM medicos WHERE usuario=?", (usuario,))
        fila = cursor.fetchone()

        if fila and bcrypt.checkpw(clave.encode(), fila[1].encode()):
            return {"acceso": True, "usuario": usuario}
        else:
            return {"acceso": False}
    except Exception:
        logging.error("Error en login", exc_info=True)
        return {"acceso": False, "error": "Error interno"}

print("Login normal seguro:", login_personal_SEGURO("dr.garcia", "clave123"))
print("Intento de bypass seguro:", login_personal_SEGURO("admin'--", "cualquiercosa"))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 6.1 MB/s eta 0:00:00
Login normal seguro: {'acceso': True, 'usuario': 'dr.garcia'}
Intento de bypass seguro: {'acceso': False, 'error': 'Usuario inválido'}


In [18]:
def diagnostico_SEGURO(paciente_id):
    try:
        sql = "SELECT nombre FROM pacientes WHERE id=?"
        print("[SEGURO] SQL generado:", sql, "con valor=", (paciente_id,))
        resultado = cursor.execute(sql, (paciente_id,)).fetchone()

        if resultado:
            return {"paciente": resultado[0], "diagnostico": "Consulta realizada"}
        else:
            return {"error": "Paciente no encontrado"}
    except Exception:
        logging.error("Error en diagnóstico", exc_info=True)
        return {"error": "Error interno"}

print("Diagnóstico seguro:", diagnostico_SEGURO(1))
print("Intento de SQL Injection bloqueado:", diagnostico_SEGURO("' OR 1=1--"))



[SEGURO] SQL generado: SELECT nombre FROM pacientes WHERE id=? con valor= (1,)
Diagnóstico seguro: {'paciente': 'Garcia', 'diagnostico': 'Consulta realizada'}
[SEGURO] SQL generado: SELECT nombre FROM pacientes WHERE id=? con valor= ("' OR 1=1--",)
Intento de SQL Injection bloqueado: {'error': 'Paciente no encontrado'}
